# E2 · Notebook 2 — Intervención preliminar: decoder de SAM 2.1 congelado vs ajustado

**NO EJECUTADO.** Solo E2. Compara el *mask decoder* **congelado** (época 0) con el **ajustado** (solo ese bloque; encoder de imagen y de prompts congelados), BCE+Dice sobre un subconjunto pequeño de train, evaluado en un subconjunto de val.

* **Referencia asistida por etiquetas, no desplegable**; DINOv3 no se usa. Preliminar: semilla 42, subconjuntos pequeños, sin test ni Bolivia.
* La hipótesis se guarda en tu Drive **antes** de cargar o entrenar el modelo (celda `prepare`).
* Los resultados reales aparecerán solo cuando tú ejecutes. Hasta revisar tus salidas, métricas e interpretación siguen **pendientes**.
* Requisito: haber ejecutado `E2_01_datos_baseline_teardown.ipynb` en tu cuenta. Runtime: **T4 GPU**.

## 1. Entorno (mismo commit que el notebook 1)

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
GITHUB_URL = 'https://github.com/roberto-chavez/auto-prompt-sam2.git'
BRANCH = '_E.V_'
PINNED_COMMIT = ''    # '' = usa el commit ya fijado en tu Drive; en la 1.ª ejecución fija la punta actual de la rama
UPDATE_CODE = False   # True SOLO para adoptar código nuevo de la rama (entonces rehaz E2 desde 'prepare')
PERSIST_ROOT = '/content/drive/MyDrive/auto-prompt-sam2'   # datos, pesos, cachés y checkpoints: SOLO aquí
REPO = Path('/content/auto-prompt-sam2')
PYTHON = Path('/content/auto-prompt-venv/bin/python')
Path(PERSIST_ROOT).mkdir(parents=True, exist_ok=True)

def sh(args, cwd=None):
    r = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if r.returncode:
        raise RuntimeError(f'{args[:3]} falló:\n{r.stdout}\n{r.stderr}')
    return r.stdout.strip()

# --- Código desde GitHub, en una revisión fijada y reproducible --------------------------------------
PIN_FILE = Path(PERSIST_ROOT) / 'e2/code_commit.txt'
existed = REPO.exists()
if not existed:
    sh(['git', 'clone', '--branch', BRANCH, GITHUB_URL, str(REPO)])
elif UPDATE_CODE:
    sh(['git', 'fetch', 'origin', BRANCH], cwd=REPO)
want = PINNED_COMMIT.strip() or ('' if UPDATE_CODE or not PIN_FILE.exists() else PIN_FILE.read_text().strip())
if UPDATE_CODE and existed and not PINNED_COMMIT.strip():
    want = sh(['git', 'rev-parse', 'FETCH_HEAD'], cwd=REPO)
if want:
    sh(['git', 'checkout', '--detach', want], cwd=REPO)
COMMIT = sh(['git', 'rev-parse', 'HEAD'], cwd=REPO)
assert not sh(['git', 'status', '--porcelain'], cwd=REPO), 'El código clonado tiene cambios locales'
if PINNED_COMMIT.strip() or UPDATE_CODE or not PIN_FILE.exists():
    PIN_FILE.parent.mkdir(parents=True, exist_ok=True)
    PIN_FILE.write_text(COMMIT)
print('Código:', GITHUB_URL, '| rama', BRANCH, '| commit', COMMIT)
print('Commit fijado en Drive:', PIN_FILE.read_text().strip())

# --- Entorno ----------------------------------------------------------------------------------------
child_env = dict(os.environ, PYTHONPATH=str(REPO / 'src'),
    AUTO_PROMPT_PERSIST_ROOT=PERSIST_ROOT, PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8', HF_HUB_DISABLE_TELEMETRY='1',
    AUTO_PROMPT_EXECUTION_CONTEXT='google_colab_user_notebook')
try:   # HF_TOKEN se lee de los secretos de Colab (activa "Acceso desde el notebook"); nunca se imprime ni se guarda
    from google.colab import userdata
    child_env['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN leído de los secretos de Colab (valor no mostrado).')
except Exception as error:
    print('HF_TOKEN no disponible (' + type(error).__name__ + '): SAM es público; solo DINOv3 lo necesita.')
LOG_DIR = Path(PERSIST_ROOT) / 'e2/notebook_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

def command(args, log_name):
    """Ejecuta un proceso mostrando su salida en vivo y guardándola en Drive."""
    with (LOG_DIR / log_name).open('w', encoding='utf-8') as log:
        process = subprocess.Popen(args, cwd=REPO, env=child_env, text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        result = process.wait()
    if result:
        raise RuntimeError(f'Proceso terminado con código {result}; revisa la salida y el log {log_name}')

if not PYTHON.is_file():
    command([sys.executable, str(REPO / 'scripts/bootstrap_colab.py'), '--venv', str(PYTHON.parent.parent)], 'bootstrap.log')
command([str(PYTHON), '-m', 'pip', 'install', '--no-deps', '-r', str(REPO / 'requirements-stage2.txt')], 'stage2_pins.log')
command([str(PYTHON), '-m', 'pip', 'check'], 'pip_check.log')
print('Entorno listo.')

In [ ]:
# Comprueba que Colab te dio una GPU (Entorno de ejecución > Cambiar tipo > T4 GPU).
command(['nvidia-smi'], 'nvidia_smi.log')
command([str(PYTHON), '-c', 'import torch; assert torch.cuda.is_available(), "NO HAY GPU: cambia el tipo de entorno a GPU"; print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)'], 'gpu_check.log')

## 2. Prerrequisitos en tu Drive

In [ ]:
# E2 lee lo que dejó TU ejecución del notebook E2_01 en TU Drive. Nada depende de archivos de otra persona.
root = Path(PERSIST_ROOT)
required = {
    'raw/acquisition (descarga Sen1Floods11)': root / 'raw/acquisition',
    'stage2/validation (datos preparados)': root / 'stage2/validation',
    'regimes/official/manifests': root / 'regimes/official/manifests',
    'regimes/official/preprocessing': root / 'regimes/official/preprocessing',
}
for name, path in required.items():
    print(('OK       ' if path.exists() else 'FALTA    ') + name)
assert all(p.exists() for p in required.values()), 'Faltan datos: ejecuta primero E2_01_datos_baseline_teardown.ipynb completo (celda de la etapa 2).'
for label, folder in (('línea base (etapa 3)', root / 'stage3/evidence'), ('teardown (etapa 4)', root / 'stage4/evidence')):
    done = sorted(p.name for p in folder.glob('*/validation.json')) if folder.exists() else []
    print(('OK       ' if done else 'AVISO    ') + label + ': ' + ('run en tu Drive' if done else 'aún no ejecutada en tu Drive (E2 puede correr, pero la evidencia de Fase A estará incompleta)'))

## 3. Hipótesis y configuración (antes de entrenar)
Tal como están en `configs/e2_intervention.yaml` del commit fijado. No se cambian tras ver resultados.

In [ ]:
print(json.dumps(json.loads((REPO / 'configs/e2_intervention.yaml').read_text()), indent=2, ensure_ascii=False))

## 4. Ejecutar E2 (`prepare`)
En orden: pruebas, verificación del hash de SAM, **guardado de la hipótesis**, paridad con el predictor de la etapa 3, época 0 (decoder congelado), entrenamiento solo del decoder con auditoría de gradientes, evaluación de la mejor época y evidencia. Imprime el tiempo por época (no estimado de antemano).

Si Colab se corta durante el entrenamiento: reconecta, ejecuta las celdas 1–2 y cambia `MODE` a `'resume'` (nunca dos escritores).

In [ ]:
MODE = 'prepare'   # 'resume' SOLO para continuar un run interrumpido
assert MODE in ('prepare', 'resume')
command([str(PYTHON), str(REPO / 'scripts/run_e2.py'), MODE, '--root', PERSIST_ROOT], MODE + '.log')

## 5. Ver resultados medidos
Todo sale de `report.json`; no hay números escritos a mano.

In [ ]:
from IPython.display import display, Image, HTML
import csv, html
EVIDENCE = Path(PERSIST_ROOT) / 'e2/evidence'
RUNS = sorted(p for p in EVIDENCE.iterdir() if p.is_dir() and (p / 'report.json').is_file())
assert RUNS, 'No hay un prepare completo. Conserva e2/notebook_logs/prepare.log.'
RUN_DIR = RUNS[-1]
REPORT = json.loads((RUN_DIR / 'report.json').read_text())
print('run_id:', REPORT['run_id'], '| estado:', REPORT['status'])
print(json.dumps({'hipotesis_previa': REPORT['hypothesis']['statement'], 'brazos': REPORT['arms'],
                  'mejor_epoca': REPORT['best_epoch'], 'verificacion_hipotesis': REPORT['hypothesis_check'],
                  'congelamiento': {k: REPORT['freezing'][k] for k in ('frozen_blocks_unchanged', 'decoder_changed')},
                  'paridad_con_etapa3': REPORT['parity_with_stage3_predictor'],
                  'subconjuntos': {k: REPORT['subsets'][k] for k in ('val_stats', 'train_stats')}},
                 indent=2, ensure_ascii=False))
for name in ('history.csv', 'per_chip_val.csv'):
    rows = list(csv.reader((RUN_DIR / name).open(encoding='utf-8', newline='')))
    print(name)
    display(HTML('<table>' + ''.join('<tr>' + ''.join('<td>' + html.escape(v) + '</td>' for v in row) + '</tr>' for row in rows) + '</table>'))
display(Image(filename=str(RUN_DIR / 'curves.png')))
for figure in sorted((RUN_DIR / 'examples').glob('*.png')):
    print(figure.name)
    display(Image(filename=str(figure)))
print('Limitaciones declaradas:'); [print(' -', item) for item in REPORT['limitations']]

## 6. Borrador del cuaderno de decisiones
Copia solo las cifras medidas a un `.md`. La **interpretación** debe escribirla el equipo.

In [ ]:
EVID = Path(PERSIST_ROOT) / 'e2/evidence'
RUN_DIR = sorted(p for p in EVID.iterdir() if p.is_dir() and (p / 'report.json').is_file())[-1]
# Borrador (solo cifras medidas) para pegar en docs/project/cuaderno_decisiones.md; la interpretación queda PENDIENTE.
command([str(PYTHON), str(REPO / 'scripts/e2_decision_log_draft.py'), str(RUN_DIR)], 'decision_log_draft.log')
print('\nArchivo:', RUN_DIR / 'e2_resultados_para_cuaderno_decisiones.md')

## 7. Recuperar en un runtime nuevo (`recover`)
Descarga primero la evidencia (sección 8). Luego *Entorno de ejecución → Desconectar y eliminar*, reconecta con T4, ejecuta **solo** las celdas 1–2 (entorno y prerrequisitos) y la celda siguiente. **No repitas `prepare`.** Recarga SAM desde tu Drive offline, verifica el hash, recarga el mejor decoder y reevalúa ambos brazos; deben coincidir (IoU agrupado a 1e-6 y confusiones por chip idénticas). Si falla, no cambies tolerancias.

In [ ]:
command([str(PYTHON), str(REPO / 'scripts/run_e2.py'), 'recover', '--root', PERSIST_ROOT], 'recover.log')

## 8. Descargar la evidencia
El ZIP incluye informe, tablas, curvas, ejemplos, predicciones, puntos, el borrador y el commit de código. **No** incluye checkpoints, pesos, tokens ni imágenes de entrenamiento.

In [ ]:
from google.colab import files
import hashlib
# Regenera el ZIP incluyendo el borrador del cuaderno de decisiones y el commit de código usado.
import zipfile
EVID = Path(PERSIST_ROOT) / 'e2/evidence'
RUN_DIR = sorted(p for p in EVID.iterdir() if p.is_dir() and (p / 'report.json').is_file())[-1]
(RUN_DIR / 'code_commit.txt').write_text(COMMIT)
target = RUN_DIR.with_name(RUN_DIR.name + '-final.zip')
with zipfile.ZipFile(target, 'w', zipfile.ZIP_DEFLATED) as archive:
    for p in sorted(RUN_DIR.rglob('*')):
        if p.is_file():
            archive.write(p, p.relative_to(RUN_DIR.parent).as_posix())
print(target, hashlib.sha256(target.read_bytes()).hexdigest())
files.download(str(target))

## 9. Qué devolver y qué NO afirmar
Devuelve: **este notebook ejecutado** (y el E2_01 ejecutado), el ZIP final de evidencia y, si hubo errores, `e2/notebook_logs/*.log`. E2 **no queda acreditada** por crear o correr estos cuadernos: falta revisar resultados, interpretarlos (incluyendo fallos y límites) y completar el cuaderno de decisiones. IA: Claude preparó código, pruebas y documentación; la comprensión y la defensa oral son tuyas.